# Tool 1: RAG_search

In [0]:

spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.rag_search(
    question STRING COMMENT 'The medical or facility question to search for semantically. 
                             Examples: "which hospitals have ICU beds", 
                             "find facilities with dialysis treatment",
                             "what equipment does Banhart Specialist Hospital have",
                             "Are there any clinics in Accra that do heart treatment?",
                             Do NOT use for counting or aggregation queries — use sql_query instead.',

    region   STRING COMMENT 'Optional. Filter results to a specific Ghana region.
                             Pass NULL to search all regions.
                             Valid values: Greater Accra Region, Ashanti Region, 
                             Western Region, Eastern Region, Central Region,
                             Northern Region, Upper East Region, Upper West Region,
                             Volta Region, Bono Region, Oti Region, Ahafo Region,
                             Bono East Region, North East Region, Savannah Region,
                             Western North Region.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Semantic search over Ghana medical facility documents using vector similarity.
         
         USE THIS TOOL WHEN the question involves:
         - What capabilities, procedures, or equipment a facility has
         - Finding facilities that offer a specific medical service
         - Questions about clinical specialties or care levels
         - Free-text descriptions of facility services
         - Any question where the answer requires understanding meaning not exact keywords
         
         DO NOT USE THIS TOOL FOR:
         - Counting facilities (use sql_query instead)
         - Filtering by structured fields like city or type (use sql_query instead)
         - Getting full details of one specific facility (use get_facility instead)
         
         RETURNS: JSON string with fields:
         - answer: natural language answer based on retrieved facilities
         - citations: list of facilities used, each with name, city, region, 
                      facility_type, lat, lon, relevance_score
         - mappable: facilities with GPS coordinates for map display
         - num_sources: number of facilities retrieved'
AS $$
import requests
import json

WORKSPACE_URL = "your-WORKSPACE_URL"
RAG_ENDPOINT  = "your-RAG_ENDPOINT"
DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

payload = {
    "dataframe_records": [
        {
            "question": question, 
            "region"  : region if region else None
        }
    ]
}

try:
    resp = requests.post(
        f"https://{WORKSPACE_URL}/serving-endpoints/{RAG_ENDPOINT}/invocations",
        headers = headers,
        json    = payload,
        timeout = 180
    )
    if resp.status_code != 200:
        return json.dumps({"error": f"RAG failed: {resp.status_code}: {resp.text}"})

    pred      = resp.json()["predictions"][0]
    citations = pred.get("citations", "[]")
    mappable  = pred.get("mappable_facilities", "[]")

    return json.dumps({
        "answer"     : pred.get("answer", ""),
        "citations"  : json.loads(citations) if isinstance(citations, str) else citations,
        "mappable"   : json.loads(mappable)  if isinstance(mappable,  str) else mappable,
        "num_sources": pred.get("num_sources", 0)
    })

except Exception as e:
    return json.dumps({"error": str(e)})
$$
""")

print("rag_search UC Function created successfully")

# Tool 2: Genie Text2SQL

In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.sql_query(
    question STRING COMMENT 'Plain English question that requires structured data analysis.
                             Examples:
                             "how many hospitals are in Greater Accra Region",
                             "How many hospitals in Accra have the ability to perform treatments related to the Heart ?",
                             "How many hospitals have cardiology?",
                             "list all public hospitals in Ashanti Region",
                             "which region has the most healthcare facilities",
                             "find regions with fewer than 3 hospitals",
                             Be specific — include filters, groupings, or conditions 
                             you want in the SQL.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Converts plain English to SQL and runs it against the Ghana facilities Delta Table.
         
         USE THIS TOOL WHEN the question involves:
         - Counting: how many, total number of
         - Filtering: list all X in Y, facilities where Z
         - Aggregating: average, sum, maximum, minimum
         - Ranking: which region has most/least, top N
         - Comparing numbers across regions or facility types
         - Any question that needs exact numbers from structured columns
                  
         RETURNS: JSON string with fields:
         - answer: natural language description of the result
         - sql_query: the SQL that was executed
         - results: list of result rows as dicts (max 20 rows)
         - num_rows: total number of rows returned'
AS $$
import requests
import json
import time

WORKSPACE_URL = "your-WORKSPACE_URL"
GENIE_SPACE_ID = "your-genie-space-id"
DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

BASE = f"https://{WORKSPACE_URL}/api/2.0/genie/spaces/{GENIE_SPACE_ID}"

try:
    # STEP 1: Start conversation
    start = requests.post(
        f"{BASE}/start-conversation",
        headers=headers,
        json={"content": question},
        timeout=30
    )

    if start.status_code != 200:
        return json.dumps({"error": f"Genie failed: {start.text}"})

    data = start.json()
    conv_id = data["conversation_id"]
    msg_id = data["message_id"]

    # STEP 2: Poll
    message = None
    for _ in range(30):
        poll = requests.get(
            f"{BASE}/conversations/{conv_id}/messages/{msg_id}",
            headers=headers,
            timeout=30
        )

        message = poll.json()
        status = message.get("status", "")

        if status == "COMPLETED":
            break
        elif status in ("FAILED", "CANCELLED"):
            return json.dumps({"error": f"Genie {status}", "raw": message})

        time.sleep(2)

    # STEP 3: Parse attachments
    attachments = message.get("attachments", [])

    sql_query = None
    description = None
    text_answer = ""

    for a in attachments:
        if "query" in a:
            sql_query = a["query"].get("query")
            description = a["query"].get("description")

        if "text" in a:
            text_answer += a["text"].get("content", "") + " "

    text_answer = text_answer.strip()

    # If no SQL, return text
    if not sql_query:
        return json.dumps({
            "answer": text_answer or "No answer returned",
            "sql_query": None,
            "results": [],
            "num_rows": 0
        })

    # STEP 4: Get results
    result_resp = requests.get(
        f"{BASE}/conversations/{conv_id}/messages/{msg_id}/query-result",
        headers=headers,
        timeout=60
    )

    if result_resp.status_code != 200:
        return json.dumps({"error": result_resp.text})

    data = result_resp.json()

    columns = [
        c["name"]
        for c in data.get("statement_response", {})
                     .get("manifest", {})
                     .get("schema", {})
                     .get("columns", [])
    ]

    rows = data.get("statement_response", {}) \
               .get("result", {}) \
               .get("data_typed_array", [])

    results = []
    for r in rows:
        values = [v.get("str", None) for v in r.get("values", [])]
        results.append(dict(zip(columns, values)))

    return json.dumps({
        "answer": text_answer or description or f"{len(results)} rows returned",
        "sql_query": sql_query,
        "results": results[:20],
        "num_rows": len(results)
    })

except Exception as e:
    return json.dumps({"error": str(e)})

$$
""")

print("sql_query function created")

# Tool 3 Get Facility Details

In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.get_facility(
    name STRING COMMENT 'Name of the specific facility to look up.
                         Can be exact or partial name.
                         Examples: "2BN Military Hospital", "Abuakwa Maternity Home".
                         Use when you already know the facility name from 
                         a previous rag_search or sql_query result.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Fetch complete structured details for one specific Ghana medical facility by name.
         
         USE THIS TOOL WHEN:
         - You found a facility name from rag_search and need its exact structured data
         - You need to verify specific claims about a named facility
         - You need the exact lat/lon of a specific facility for geospatial calculations
         - You need to cross-reference a facility found in one tool with structured data
         - A user asks specifically about one named facility
         
         DO NOT USE THIS TOOL FOR:
         - Searching across multiple facilities (use rag_search instead)
         - Counting or aggregating (use sql_query instead)
         
         RETURNS: JSON string with fields:
         - found: true/false
         - facility: dict with all columns for this facility
         - lat: latitude as float (0.0 if unknown)
         - lon: longitude as float (0.0 if unknown)'
AS $$
import json

import requests, time

WORKSPACE_URL = "your-WORKSPACE_URL"
GENIE_SPACE_ID = "your-genie-space-id"
DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json"
}

BASE = f"https://{WORKSPACE_URL}/api/2.0/genie/spaces/{GENIE_SPACE_ID}"

question = (
    f"Get all details for the facility named '{name}'. "
    f"Return name, facilityTypeId, operatorTypeId, address_city, "
    f"address_stateOrRegion, numberDoctors, capacity, lat, lon, "
    f"specialties, capability, description."
)

try:
    start   = requests.post(f"{BASE}/start-conversation",
                            headers=headers, json={"content": question}, timeout=30)
    
    if start.status_code != 200:
        return json.dumps({"error": f"Failed to start: {start.text}"})
        
    conv_id = start.json()["conversation_id"]
    msg_id  = start.json()["message_id"]
    message = None

    for _ in range(30):
        poll   = requests.get(f"{BASE}/conversations/{conv_id}/messages/{msg_id}",
                              headers=headers, timeout=30)
        message = poll.json()
        status = message.get("status")
        
        if status == "COMPLETED":
            break
        elif status in ("FAILED", "CANCELLED"):
            return json.dumps({"error": f"Genie query failed: {status}"})
            
        time.sleep(2)
    
    attachments = message.get("attachments", [])
    sql_query = None

    for a in attachments:
        if "query" in a:
            sql_query = a["query"].get("query")

    # FIX #2: Return correct schema if no SQL
    if not sql_query:
        return json.dumps({
            "found": False,
            "facility": {},
            "error": "No SQL generated"
        })

    # STEP 4: Get results
    result_resp = requests.get(
        f"{BASE}/conversations/{conv_id}/messages/{msg_id}/query-result",
        headers=headers,
        timeout=60
    )

    if result_resp.status_code != 200:
        return json.dumps({"error": result_resp.text})

    data = result_resp.json()

    columns = [
        c["name"]
        for c in data.get("statement_response", {})
                     .get("manifest", {})
                     .get("schema", {})
                     .get("columns", [])
    ]

    rows = data.get("statement_response", {}) \
               .get("result", {}) \
               .get("data_typed_array", [])

    results = []
    for r in rows:
        values = [v.get("str", None) for v in r.get("values", [])]
        results.append(dict(zip(columns, values)))

    # FIX #1: Correct Indentation
    if results:
        f = results[0]
        
        # FIX #3: Safely convert to float
        try:
            lat = float(f.get("lat")) if f.get("lat") else 0.0
        except (ValueError, TypeError):
            lat = 0.0
            
        try:
            lon = float(f.get("lon")) if f.get("lon") else 0.0
        except (ValueError, TypeError):
            lon = 0.0

        return json.dumps({
            "found"   : True,
            "facility": f,
            "lat"     : lat,
            "lon"     : lon
        })
        
    return json.dumps({"found": False, "facility": {}})

except Exception as e:
    return json.dumps({"error": str(e)})

$$
""")
print("created get_facility function")


# Tool 4: External Data


In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.external_data(
    data_type STRING COMMENT 'The type of reference data to retrieve. Must be one of:
                              "population"      - Ghana region population figures (2021 census)
                              "area"            - Region area in square kilometres
                              "capital"         - Regional capital city name and GPS coordinates
                              "who_standards"   - WHO minimum healthcare benchmarks
                              "health_context"  - Ghana health system context and averages
                              "derived_metrics" - Pre-calculated: hospitals needed, doctors needed
                                                  beds needed per region based on WHO standards
                              "all_regions"     - Population + area + capital for all 16 regions',

    region   STRING COMMENT 'Optional. Specific Ghana region to get data for.
                             Pass NULL to get data for all regions.
                             Example: "Northern Region", "Upper East Region".
                             Only applicable for data_type: population, area, capital, 
                             derived_metrics.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Returns static reference data about Ghana regions and WHO healthcare standards.
         No API call — instant response from pre-loaded reference data.
         
         USE THIS TOOL WHEN:
         - You need Ghana region populations for per-capita calculations
         - Comparing facility counts against WHO minimum standards
         - You need coordinates of a regional capital for geospatial queries
         - Calculating healthcare deficits (hospitals needed vs actual)
         - Any question involving "per capita", "per 100,000", "relative to population"
         - Identifying medical deserts using WHO benchmarks
         
         ALWAYS CALL THIS BEFORE doing per-capita calculations —
         do not estimate or guess population figures.
         
         COMMON PATTERN:
         1. Call sql_query to get facility counts per region
         2. Call external_data("derived_metrics") to get hospitals_needed_who per region
         3. Compare actual vs needed to identify underserved regions
         
         WHO STANDARDS included:
         - Minimum 1 hospital per 100,000 people
         - Minimum 1 physician per 1,000 people  
         - Minimum 1 hospital bed per 1,000 people
         
         RETURNS: JSON string — structure depends on data_type requested.'
AS $$
import json

GHANA_DATA = {
    "populations": {
        "Greater Accra Region": 5455692, "Ashanti Region": 5440463,
        "Eastern Region": 2925653,       "Central Region": 2859821,
        "Northern Region": 2310939,      "Western Region": 2060585,
        "Volta Region": 1652772,         "Upper East Region": 1301226,
        "Bono Region": 1208649,          "Bono East Region": 1202739,
        "Upper West Region": 901502,      "Western North Region": 880921,
        "Oti Region": 747248,            "Savannah Region": 653266,
        "North East Region": 650856,     "Ahafo Region": 564662,
    },
    "area_km2": {
        "Savannah Region": 35862,      "Northern Region": 25448,
        "Ashanti Region": 24389,       "Bono East Region": 23257,
        "Eastern Region": 19323,       "Upper West Region": 18476,
        "Volta Region": 9504,          "Western Region": 13847,
        "North East Region": 9074,     "Upper East Region": 8842,
        "Bono Region": 11107,          "Oti Region": 11066,
        "Western North Region": 10074, "Central Region": 9826,
        "Ahafo Region": 5193,          "Greater Accra Region": 3245,
    },
    "capitals": {
        "Greater Accra Region": {"city": "Accra",        "lat": 5.6037,  "lon": -0.1870},
        "Ashanti Region":       {"city": "Kumasi",       "lat": 6.6885,  "lon": -1.6244},
        "Western Region":       {"city": "Sekondi",      "lat": 4.9341,  "lon": -1.7097},
        "Eastern Region":       {"city": "Koforidua",    "lat": 6.0940,  "lon": -0.2574},
        "Central Region":       {"city": "Cape Coast",   "lat": 5.1054,  "lon": -1.2466},
        "Northern Region":      {"city": "Tamale",       "lat": 9.4008,  "lon": -0.8393},
        "Upper East Region":    {"city": "Bolgatanga",   "lat": 10.7856, "lon": -0.8514},
        "Upper West Region":    {"city": "Wa",           "lat": 10.0601, "lon": -2.5099},
        "Volta Region":         {"city": "Ho",           "lat": 6.6011,  "lon":  0.4703},
        "Bono Region":          {"city": "Sunyani",      "lat": 7.3349,  "lon": -2.3269},
        "Bono East Region":     {"city": "Techiman",     "lat": 7.5833,  "lon": -1.9333},
        "Ahafo Region":         {"city": "Goaso",        "lat": 6.8000,  "lon": -2.5167},
        "Oti Region":           {"city": "Dambai",       "lat": 8.0667,  "lon":  0.1833},
        "Savannah Region":      {"city": "Damongo",      "lat": 9.0833,  "lon": -1.8167},
        "North East Region":    {"city": "Nalerigu",     "lat": 10.5306, "lon": -0.3686},
        "Western North Region": {"city": "Sefwi Wiawso", "lat": 6.1667,  "lon": -2.4833},
    },
    "who_standards": {
        "physicians_per_1000": 1.0,
        "hospital_beds_per_1000": 1.0,
        "hospitals_per_100000": 1.0,
        "nurses_per_1000": 3.0
    }
}

who = GHANA_DATA["who_standards"]

if data_type == "population":
    if region:
        pop = GHANA_DATA["populations"].get(region)
        return json.dumps({"region": region, "population": pop} if pop
                          else {"error": f"Unknown region: {region}"})
    return json.dumps({"populations": GHANA_DATA["populations"]})

elif data_type == "area":
    if region:
        area = GHANA_DATA["area_km2"].get(region)
        return json.dumps({"region": region, "area_km2": area} if area
                          else {"error": f"Unknown region: {region}"})
    return json.dumps({"areas": GHANA_DATA["area_km2"]})

elif data_type == "capital":
    if region:
        cap = GHANA_DATA["capitals"].get(region)
        return json.dumps({"region": region, **cap} if cap
                          else {"error": f"Unknown region: {region}"})
    return json.dumps({"capitals": GHANA_DATA["capitals"]})

elif data_type == "who_standards":
    return json.dumps({"who_standards": who})

elif data_type == "derived_metrics":
    metrics = {}
    for r, pop in GHANA_DATA["populations"].items():
        area = GHANA_DATA["area_km2"].get(r, 1)
        metrics[r] = {
            "population"           : pop,
            "area_km2"             : area,
            "pop_density_per_km2"  : round(pop / area, 1),
            "hospitals_needed_who" : round((pop / 100000) * who["hospitals_per_100000"]),
            "doctors_needed_who"   : round((pop / 1000)   * who["physicians_per_1000"]),
            "beds_needed_who"      : round((pop / 1000)   * who["hospital_beds_per_1000"]),
        }
    if region:
        return json.dumps({"region": region,
                           "metrics": metrics.get(region, {})})
    return json.dumps({"derived_metrics": metrics})

return json.dumps({"error": f"Unknown data_type: {data_type}"})
$$
""")

print("external_data function created")

# Tool 5: Find nearby facilities

In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.find_nearby_facilities(
    center_lat   DOUBLE  COMMENT 'Latitude of the center point to search around.
                                  Get this from external_data("capital") for regional
                                  capitals, or from get_facility() for named facilities.',
    center_lon   DOUBLE  COMMENT 'Longitude of the center point to search around.',
    radius_km    DOUBLE  COMMENT 'Search radius in kilometres. 
                                  Typical values: 10 (urban), 50 (regional), 100 (national).
                                  WHO recommends emergency care within 50km.',
    condition    STRING  COMMENT 'Optional. Medical condition, procedure, or capability to filter by.
                                  Pass NULL to find all facilities within radius.
                                  Examples: "dialysis", "surgery", "maternity", "ICU", "emergency"
                                  This does a keyword match against capability, procedure, 
                                  specialties columns.',
    facility_type STRING COMMENT 'Optional. Filter by facility type.
                                  Pass NULL for all types.
                                  Valid values: hospital, clinic, pharmacy, doctor, dentist'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Find all facilities within a radius of a point that match an optional condition.
         Combines geospatial distance calculation with capability filtering in one call.

         USE THIS TOOL WHEN:
         - "hospitals within X km of Y"
         - "facilities treating Z near location"
         - "what healthcare is accessible from this point"
         - "is there a hospital within 50km of this area"

         WORKFLOW:
         1. Get center coordinates from external_data or get_facility first
         2. Call this tool with those coordinates
         3. Results are already filtered by radius and sorted by distance

         RETURNS: JSON with
         - facilities: list sorted by distance_km (closest first)
                       each has name, city, region, facility_type, 
                       lat, lon, distance_km, matched_capabilities
         - count: total facilities found within radius
         - radius_km: the search radius used
         - nearest: the single closest matching facility'
AS $$
import requests
import json
import math

# WORKSPACE_URL = "your-WORKSPACE_URL"
# WAREHOUSE_ID = "your-WAREHOUSE_ID"
# DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

def haversine(lat1, lon1, lat2, lon2):
    R    = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a    = (math.sin(dlat/2)**2 +
            math.cos(math.radians(lat1)) *
            math.cos(math.radians(lat2)) *
            math.sin(dlon/2)**2)
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

try:
    BASE_URL = f"https://{WORKSPACE_URL}/api/2.0/sql/statements"

    # 1. Build the exact SQL Query    
    condition_clause = ""
    if condition:
        c = condition.lower()
        condition_clause = (
            f"AND ("
            f"  exists(capability, x -> x ilike '%{c}%') OR "
            f"  exists(procedure, x -> x ilike '%{c}%') OR "
            f"  exists(specialties, x -> x ilike '%{c}%')"
            f")"
        )

    type_clause = ""
    if facility_type:
        type_clause = f"AND facilityTypeId = '{facility_type}'"

    sql_query = (
        f"SELECT name, facilityTypeId, operatorTypeId, address_city, "
        f"address_stateOrRegion, lat, lon, capability, procedure, specialties "
        f"FROM facilities3 "
        f"WHERE lat IS NOT NULL AND lon IS NOT NULL "
        f"AND lat != 0 AND lon != 0 "
        f"{condition_clause} {type_clause}"
    )

    # 2. Execute directly on the SQL Warehouse
    payload = {
        "warehouse_id": WAREHOUSE_ID,
        "statement": sql_query,
        "wait_timeout": "30s" # Wait synchronously for up to 30 seconds
    }

    start = requests.post(BASE_URL, headers=headers, json=payload, timeout=40)
    
    if start.status_code != 200:
        return json.dumps({"error": f"SQL Execution failed: {start.text}", "count": 0})

    data = start.json()
    status = data.get("status", {}).get("state")
    statement_id = data.get("statement_id")

    # 3. Poll if it didn't finish within the 30s wait_timeout (rare, but safe)
    while status in ("PENDING", "RUNNING"):
        time.sleep(1)
        poll = requests.get(f"{BASE_URL}/{statement_id}", headers=headers, timeout=10)
        data = poll.json()
        status = data.get("status", {}).get("state")

    if status != "SUCCEEDED":
        return json.dumps({"error": f"Query failed with status: {status}", "count": 0})

    # 4. Parse Results
    # The SQL API returns a much simpler JSON format than Genie
    manifest = data.get("manifest", {})
    columns = [c["name"] for c in manifest.get("schema", {}).get("columns", [])]
    
    # data_array is a simple list of lists: [["value1", "value2"], ["value1", "value2"]]
    rows = data.get("result", {}).get("data_array", [])
    
    all_facilities = [dict(zip(columns, r)) for r in rows]

    # 5. Filter by radius locally
    nearby = []
    for f in all_facilities:
        try:
            lat_str = f.get("lat")
            lon_str = f.get("lon")
            if not lat_str or not lon_str:
                continue
                
            f_lat = float(lat_str)
            f_lon = float(lon_str)
            
            dist = haversine(center_lat, center_lon, f_lat, f_lon)
            if dist <= radius_km:
                nearby.append({
                    "name"                : f.get("name"),
                    "facility_type"       : f.get("facilityTypeId"),
                    "city"                : f.get("address_city"),
                    "region"              : f.get("address_stateOrRegion"),
                    "lat"                 : f_lat,
                    "lon"                 : f_lon,
                    "distance_km"         : round(dist, 1),
                    "matched_capabilities": str(f.get("capability", ""))[:200]
                })
        except (ValueError, TypeError):
            continue

    # Sort by distance (closest first)
    nearby = sorted(nearby, key=lambda x: x["distance_km"])

    return json.dumps({
        "facilities": nearby,
        "count"     : len(nearby),
        "radius_km" : radius_km,
        "nearest"   : nearby[0] if nearby else None,
        "condition" : condition,
        "sql_used"  : sql_query
    })

except Exception as e:
    return json.dumps({"error": str(e), "count": 0})
$$
""")

print("find_facilities_near Function created")

# Tool 6: Find cold spots

In [0]:
spark.sql("""
CREATE OR REPLACE FUNCTION workspace.default.find_cold_spots(
    procedure_or_capability STRING COMMENT 'The critical procedure or capability to check coverage for.
                                           Examples: "surgery", "emergency care", "dialysis",
                                           "maternity", "ICU", "blood transfusion", "caesarean".
                                           The tool checks which regions have NO facility
                                           offering this within the coverage radius.',

    coverage_radius_km      DOUBLE  COMMENT 'The radius in km that defines "accessible".
                                            WHO recommends 50km for emergency care.
                                            Use 50 for emergency services.
                                            Use 100 for specialist services.
                                            Use 25 for primary care.
                                            
                                            And, IF THE USER ASKS FOR TRAVEL TIME: 
                                            Assume an average regional travel speed of 50 km/h in Ghana. 
                                            For example: "1 hour" = 50.0 km. "2 hours" = 100.0 km. "30 mins" = 25.0 km.
                                            Translate the time into kilometers before calling this tool.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Identifies geographic cold spots — regions where a critical medical procedure
         or capability is absent or inaccessible within the coverage radius.
         Uses regional capitals as representative population center points.

         USE THIS TOOL WHEN:
         - "where is X procedure not available within Y km"
         - "find cold spots for [capability]"
         - "which regions lack access to [procedure]"
         - "geographic gaps in [service] coverage"
         - "medical deserts for [condition]"

         METHODOLOGY:
         1. Gets all facilities offering the procedure with their GPS coordinates
         2. For each of the 16 regional capitals, checks if any matching
            facility exists within coverage_radius_km
         3. Regions with NO matching facility within radius = cold spots

         RETURNS: JSON with
         - cold_spots: list of regions with no coverage
                       each has region, capital, population, 
                       nearest_facility (name + distance if exists)
         - covered_regions: list of regions that have coverage
         - coverage_pct: percentage of regions covered
         - total_population_in_cold_spots: sum of populations in uncovered regions
         - procedure: the procedure checked'
AS $$
import requests
import json
import math
import time

# WORKSPACE_URL = "your-WORKSPACE_URL"
# WAREHOUSE_ID = "your-WAREHOUSE_ID"
# DATABRICKS_TOKEN="your-DATABRICKS_TOKEN"

headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type" : "application/json"
}

REGION_CAPITALS = {
    "Greater Accra Region": {"city": "Accra",        "lat": 5.6037,  "lon": -0.1870, "pop": 5455692},
    "Ashanti Region":       {"city": "Kumasi",       "lat": 6.6885,  "lon": -1.6244, "pop": 5440463},
    "Eastern Region":       {"city": "Koforidua",    "lat": 6.0940,  "lon": -0.2574, "pop": 2925653},
    "Central Region":       {"city": "Cape Coast",   "lat": 5.1054,  "lon": -1.2466, "pop": 2859821},
    "Northern Region":      {"city": "Tamale",       "lat": 9.4008,  "lon": -0.8393, "pop": 2310939},
    "Western Region":       {"city": "Sekondi",      "lat": 4.9341,  "lon": -1.7097, "pop": 2060585},
    "Volta Region":         {"city": "Ho",           "lat": 6.6011,  "lon":  0.4703, "pop": 1652772},
    "Upper East Region":    {"city": "Bolgatanga",   "lat": 10.7856, "lon": -0.8514, "pop": 1301226},
    "Bono Region":          {"city": "Sunyani",      "lat": 7.3349,  "lon": -2.3269, "pop": 1208649},
    "Bono East Region":     {"city": "Techiman",     "lat": 7.5833,  "lon": -1.9333, "pop": 1202739},
    "Upper West Region":    {"city": "Wa",           "lat": 10.0601, "lon": -2.5099, "pop": 901502},
    "Western North Region": {"city": "Sefwi Wiawso", "lat": 6.1667,  "lon": -2.4833, "pop": 880921},
    "Oti Region":           {"city": "Dambai",       "lat": 8.0667,  "lon":  0.1833, "pop": 747248},
    "Savannah Region":      {"city": "Damongo",      "lat": 9.0833,  "lon": -1.8167, "pop": 653266},
    "North East Region":    {"city": "Nalerigu",     "lat": 10.5306, "lon": -0.3686, "pop": 650856},
    "Ahafo Region":         {"city": "Goaso",        "lat": 6.8000,  "lon": -2.5167, "pop": 564662},
}

def haversine(lat1, lon1, lat2, lon2):
    R    = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a    = (math.sin(dlat/2)**2 +
            math.cos(math.radians(lat1)) *
            math.cos(math.radians(lat2)) *
            math.sin(dlon/2)**2)
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

try:
    # 1. Build SQL Statement (WITH THE ARRAY FIX)
    proc_lower = procedure_or_capability.lower()
    sql_query = (
        f"SELECT name, address_stateOrRegion, lat, lon "
        f"FROM facilities3 "
        f"WHERE lat IS NOT NULL AND lon IS NOT NULL "
        f"AND lat != 0 AND lon != 0 "
        f"AND ("
        f"  exists(capability, x -> x ilike '%{proc_lower}%') OR "
        f"  exists(procedure, x -> x ilike '%{proc_lower}%') OR "
        f"  exists(specialties, x -> x ilike '%{proc_lower}%')"
        f")"
    )

    # 2. Call SQL Execution API
    payload = {
        "warehouse_id": WAREHOUSE_ID,
        "statement": sql_query,
        "wait_timeout": "30s"
    }
    
    resp = requests.post(f"https://{WORKSPACE_URL}/api/2.0/sql/statements", 
                         headers=headers, json=payload, timeout=40)
    
    if resp.status_code != 200:
        return json.dumps({"error": f"SQL API Error: {resp.text}"})

    data = resp.json()
    
    # Check status
    status = data.get("status", {}).get("state")
    if status != "SUCCEEDED":
        statement_id = data.get("statement_id")
        for _ in range(10):
            time.sleep(1)
            poll = requests.get(f"https://{WORKSPACE_URL}/api/2.0/sql/statements/{statement_id}", 
                                headers=headers)
            data = poll.json()
            if data.get("status", {}).get("state") == "SUCCEEDED":
                break

    # 3. Parse Results
    manifest = data.get("manifest", {})
    columns = [c["name"] for c in manifest.get("schema", {}).get("columns", [])]
    rows = data.get("result", {}).get("data_array", [])
    
    matching_facilities = [dict(zip(columns, r)) for r in rows]

    # 4. Geospatial Analysis
    cold_spots = []
    covered_regions = []
    cold_spot_pop = 0

    for region, capital in REGION_CAPITALS.items():
        cap_lat = capital["lat"]
        cap_lon = capital["lon"]
        
        nearest = None
        nearest_dist = float("inf")

        for f in matching_facilities:
            try:
                f_lat, f_lon = float(f["lat"]), float(f["lon"])
                dist = haversine(cap_lat, cap_lon, f_lat, f_lon)
                if dist < nearest_dist:
                    nearest_dist = dist
                    nearest = f
            except: continue

        if nearest_dist > coverage_radius_km:
            cold_spots.append({
                "region": region,
                "capital": capital["city"],
                "population": capital["pop"],
                "nearest_facility": nearest.get("name") if nearest else "None in dataset",
                "nearest_dist_km": round(nearest_dist, 1) if nearest else None
            })
            cold_spot_pop += capital["pop"]
        else:
            covered_regions.append(region)

    # 5. Final Output
    return json.dumps({
        "procedure": procedure_or_capability,
        "radius_km": coverage_radius_km,
        "cold_spots": sorted(cold_spots, key=lambda x: x["population"], reverse=True),
        "num_cold_spots": len(cold_spots),
        "coverage_pct": round(len(covered_regions) / 16 * 100, 1),
        "total_population_affected": cold_spot_pop
    })

except Exception as e:
    return json.dumps({"error": str(e)})
$$
""")

print("find_cold_spots UC Function created")